In [36]:
import pandas as pd
import xgboost as xgb

df = pd.read_parquet('stocks.parquet')

df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")

features = df.columns.difference(['id', 'date', 'ret_eom', 'gvkey', 'iid', 'excntry',
    'year', 'month', 'char_date', 'char_eom', 'stock_ret']).tolist()

# Initial training and validation periods
initial_train_end = pd.to_datetime("2012-12-31")
validation_end    = pd.to_datetime("2014-12-31")
end_date          = pd.to_datetime("2025-05-31")

# Collect predictions
predictions = []

while validation_end < end_date:

    # Masks
    train_mask = (df["date"] <= initial_train_end)
    valid_mask = (df["date"] > initial_train_end) & (df["date"] <= validation_end)
    test_mask  = (df["date"] > validation_end) & (df["date"] <= validation_end + pd.DateOffset(years=1))

    train_X, train_y = df.loc[train_mask, features], df.loc[train_mask, "stock_ret"]
    valid_X, valid_y = df.loc[valid_mask, features], df.loc[valid_mask, "stock_ret"]
    test_X           = df.loc[test_mask, features]

    dtrain = xgb.DMatrix(train_X, label=train_y)
    dvalid = xgb.DMatrix(valid_X, label=valid_y)
    dtest  = xgb.DMatrix(test_X)

    params = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "max_depth": 12,
        "eta": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "seed": 42,
        "tree_method": "gpu_hist",   # 🚀 enables GPU acceleration
        "predictor": "gpu_predictor" # (optional) ensures GPU is used for prediction too
    }

    model = xgb.train(
        params,
        dtrain,
        num_boost_round=1000,
        evals=[(dtrain, "train"), (dvalid, "valid")],
        early_stopping_rounds=50,
        verbose_eval=False,
    )

    # Predictions for test window
    preds = model.predict(dtest)

    # Save with stock_id + date
    tmp = df.loc[test_mask, ["gvkey", "date", "stock_ret"]].copy()
    tmp["predicted_return"] = preds
    predictions.append(tmp)

    # Expand window by 1 year
    initial_train_end += pd.DateOffset(years=1)
    validation_end += pd.DateOffset(years=1)


pred_df = pd.concat(predictions, ignore_index=True)



c:\Users\shoai\anaconda3\envs\MLCOURSE\lib\site-packages\xgboost\core.py:158: UserWarning: [16:26:00] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
c:\Users\shoai\anaconda3\envs\MLCOURSE\lib\site-packages\xgboost\core.py:158: UserWarning: [16:26:00] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "predictor" } are not used.

  warnings.warn(smsg, UserWarning)
c:\Users\shoai\anaconda3\envs\MLCOURSE\lib\site-packages\xgboost\core.py:158: UserWarning: [16:26:04] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-

In [69]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Ensure 'date' is datetime
pred_df["date"] = pd.to_datetime(pred_df["date"], errors='coerce')

# Filter for 2016
pred_df_2016 = pred_df[pred_df["date"].dt.year == 2016]

# Compute metrics for 2016
mse = mean_squared_error(pred_df_2016["stock_ret"], pred_df_2016["predicted_return"])
rmse = np.sqrt(mse)
mae = mean_absolute_error(pred_df_2016["stock_ret"], pred_df_2016["predicted_return"])
r2  = r2_score(pred_df_2016["stock_ret"], pred_df_2016["predicted_return"])

print(f"Performance for 2016 predictions:")
print(f"  MSE  = {mse:.6f}")
print(f"  RMSE = {rmse:.6f}")
print(f"  MAE  = {mae:.6f}")
print(f"  R²   = {r2:.6f}")

pred_df_2016


Performance for 2016 predictions:
  MSE  = 4.620544
  RMSE = 2.149545
  MAE  = 0.148424
  R²   = -87.925981


,gvkey,date,stock_ret,predicted_return
65853,1096.0,2016-01-29,-0.025970,0.005332
65854,1186.0,2016-01-29,0.126779,0.008721
65855,1262.0,2016-01-29,-0.068678,0.021070
65856,1828.0,2016-01-29,0.073554,0.017645
65857,2055.0,2016-01-29,0.342691,0.015598
...,...,...,...,...
129474,179583.0,2016-12-30,-0.058411,0.006770
129475,184514.0,2016-12-30,-0.006024,0.010779
129476,184500.0,2016-12-30,0.072424,0.006113
129477,184259.0,2016-12-30,-0.041667,0.012365


In [67]:
import joblib

# Example: load the 2015 model
model_2016 = joblib.load("xgboost_model_2015.joblib")


import pandas as pd

# Load your main dataset
df = pd.read_parquet("stocks.parquet")
df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")

# Features must match the training step
features = df.columns.difference([
    'id','date','ret_eom','gvkey','iid','excntry',
    'year','month','char_date','char_eom','stock_ret'
]).tolist()

# Filter the test year (e.g., 2015)
df_2016 = df[df["date"].dt.year == 2016]
X_2016 = df_2016[model_2016.get_booster().feature_names]
print()
y_2016 = df_2016["stock_ret"]   # for comparison later

# Predict with the loaded model
preds_2016 = model_2016.predict(X_2016)

# Attach predictions to DataFrame
results_2016 = df_2016[["gvkey","date"]].copy()
results_2016["actual_return"] = y_2016.values
results_2016["predicted_return"] = preds_2016

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

mse  = mean_squared_error(y_2016, preds_2016)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_2016, preds_2016)
r2   = r2_score(y_2016, preds_2016)

print(f"Performance for 2015 model:")
print(f"  MSE  = {mse:.6f}")
print(f"  RMSE = {rmse:.6f}")
print(f"  MAE  = {mae:.6f}")
print(f"  R²   = {r2:.6f}")





c:\Users\shoai\anaconda3\envs\MLCOURSE\lib\site-packages\xgboost\core.py:158: UserWarning: [16:47:17] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
c:\Users\shoai\anaconda3\envs\MLCOURSE\lib\site-packages\xgboost\core.py:158: UserWarning: [16:47:17] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- 

Performance for 2015 model:
  MSE  = 0.231795
  RMSE = 0.481451
  MAE  = 0.149137
  R²   = -3.461082


In [76]:
import joblib
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load your main dataset
df = pd.read_parquet("stocks.parquet")
df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")

# Store all results
all_results = []

for year in range(2015, 2026):  # 2015 → 2025
    model_year = year - 1
    model_path = f"xgboost_model_{model_year}.joblib"

    try:
        model = joblib.load(model_path)
    except FileNotFoundError:
        print(f"⚠️ Model for {model_year} not found, skipping {year}.")
        continue

    df_year = df[df["date"].dt.year == year]
    if df_year.empty:
        print(f"⚠️ No data for {year}, skipping.")
        continue

    X_year = df_year[model.get_booster().feature_names]
    y_year = df_year["stock_ret"]

    preds = model.predict(X_year)

    # Collect results
    results_year = df_year[["gvkey", "date"]].copy()
    results_year["predicted_return"] = preds
    results_year["actual_return"] = y_year.values
    results_year["model_used"] = model_year
    all_results.append(results_year)

# Combine all predictions into one DataFrame
final_results = pd.concat(all_results, ignore_index=True)

# Keep only gvkey, date, and predicted_return (per your requirement)
final_output = final_results[["gvkey", "date", "predicted_return"]]

print("\n✅ Final predictions dataframe created with shape:", final_output.shape)
print(final_output.head())


c:\Users\shoai\anaconda3\envs\MLCOURSE\lib\site-packages\xgboost\core.py:158: UserWarning: [17:54:36] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
c:\Users\shoai\anaconda3\envs\MLCOURSE\lib\site-packages\xgboost\core.py:158: UserWarning: [17:54:37] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
c:\Users\shoai\anaconda3\envs\MLCOURSE\lib\site-packages\xgboost\core.py:158: UserWarn


✅ Final predictions dataframe created with shape: (667275, 3)
    gvkey       date  predicted_return
0  1096.0 2015-01-30          0.008775
1  1186.0 2015-01-30         -0.019760
2  1262.0 2015-01-30          0.011259
3  1828.0 2015-01-30          0.005781
4  2055.0 2015-01-30         -0.007752


c:\Users\shoai\anaconda3\envs\MLCOURSE\lib\site-packages\xgboost\core.py:158: UserWarning: [17:54:40] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\common\error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
